# Steps 7 and 9, resumed: checker and metrics (no LLM generation)

**Author:** Michael Dong  
**Project:** PlainMed (COSC 72/272 final project)  
**Date:** November 15, 2026

**Why this notebook exists:** the first run of `09_check_and_test` finished both LLM test runs (saved on Drive), then Colab's free tier disconnected it during the checker, whose results were only on Colab's disk. This notebook picks up from there **without loading the LLM**:
1. copies the validation outputs (Step 4) and the test outputs (Step 9) from Drive,
2. writes the baseline outputs for the test set,
3. runs the faithfulness checker on every main system, **saving each file to Drive as soon as it is done** (a disconnect loses at most one file, and re-running skips finished files),
4. scores every system with all metrics, including real BERTScore, saving to Drive after each,
5. zips everything into one download.
6. (optional, last) runs the checker on the six prompt-sensitivity runs, for the report.

**Input:** `MyDrive/plainmed/step4/` and `MyDrive/plainmed/step9/`.  
**Output:** `MyDrive/plainmed/step9/` (checked outputs, results, and `plainmed_step9.zip`).

**Before you start:** Runtime → Change runtime type → **T4 GPU**, then Runtime → Run all. Keep this tab active: Colab's free tier disconnects sessions it thinks are idle.

### 1. GPU, Drive and the latest code

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')
import os, subprocess
STEP4 = '/content/drive/MyDrive/plainmed/step4'
RUN_DIR = '/content/drive/MyDrive/plainmed/step9'
os.makedirs(RUN_DIR, exist_ok=True)
if os.path.exists('/content/plainmed'):
    subprocess.run(['git', '-C', '/content/plainmed', 'pull', '-q'], check=True)
else:
    if subprocess.run(['git', 'clone', '-q', '--depth', '1', 'https://github.com/dongm58/plainmed', '/content/plainmed']).returncode != 0:
        from google.colab import userdata
        subprocess.run(['git', 'clone', '-q', '--depth', '1', f"https://{userdata.get('GH_TOKEN')}@github.com/dongm58/plainmed", '/content/plainmed'], check=True)
%cd /content/plainmed
!git log -1 --format='commit %h: %s'

### 2. Install and build the data

In [ ]:
!pip -q install -r requirements-colab.txt && pip -q install -e . --no-deps
!python scripts/01_data.py > /tmp/data_log.txt && head -2 /tmp/data_log.txt

### 3. Bring in the saved LLM outputs and write the test baselines

In [ ]:
!mkdir -p outputs/llm_zeroshot outputs/llm_fewshot
!cp $STEP4/outputs/llm_zeroshot/cochrane_val.jsonl outputs/llm_zeroshot/ && cp $STEP4/outputs/llm_fewshot/cochrane_val.jsonl outputs/llm_fewshot/
!cp $RUN_DIR/outputs/llm_zeroshot/cochrane_test.jsonl outputs/llm_zeroshot/ && cp $RUN_DIR/outputs/llm_fewshot/cochrane_test.jsonl outputs/llm_fewshot/
!cp -r $STEP4/outputs/prompt_sensitivity outputs/
!python scripts/03_baselines.py --split test > /tmp/baselines_log.txt && tail -1 /tmp/baselines_log.txt
!wc -l outputs/llm_*/cochrane_*.jsonl outputs/extractive/cochrane_test.jsonl

### 4. Faithfulness checker on the main systems (saved to Drive file by file)
If Colab disconnects, run cells 1–3 again and then this cell: finished files are skipped.

In [ ]:
FILES = ' '.join(f'outputs/{s}/cochrane_{sp}.jsonl' for s in ['llm_fewshot', 'llm_zeroshot', 'extractive', 'reference'] for sp in ['test', 'val'])
!python scripts/07_check.py --batch-size 64 --out-root $RUN_DIR --inputs $FILES
!mkdir -p checked && cp -r $RUN_DIR/checked/* checked/ && mkdir -p results && cp $RUN_DIR/results/checker_summary.csv results/

### 5. All metrics with real BERTScore (saved to Drive after each system)

In [ ]:
for split in ['test', 'val']:
    for system in ['llm_fewshot', 'llm_zeroshot', 'extractive', 'copy_source', 'reference']:
        !python scripts/02_score.py --system {system} --dataset cochrane --split {split} --bertscore real
        !mkdir -p $RUN_DIR/results/scores && cp results/summary.csv $RUN_DIR/results/ && cp results/scores/*.csv $RUN_DIR/results/scores/

### 6. Zip everything and download it
This downloads `plainmed_step9.zip`. A copy is also saved in `MyDrive/plainmed/step9/`.

In [ ]:
!cd /content/plainmed && zip -qr /content/plainmed_step9.zip outputs results checked && cp /content/plainmed_step9.zip $RUN_DIR/ && ls -lh /content/plainmed_step9.zip
import pandas as pd
print(pd.read_csv('results/checker_summary.csv')[['file', 'n_outputs', 'share_sentences_flagged', 'share_sentences_number_mismatch']].round(3).to_string(index=False))
from google.colab import files
files.download('/content/plainmed_step9.zip')

### 7. Optional, for the report: checker on the six prompt-sensitivity runs
Only run this once the zip above is downloaded. It also saves file by file to Drive.

In [ ]:
import glob
SENS = ' '.join(sorted(glob.glob('outputs/prompt_sensitivity/*.jsonl')))
!python scripts/07_check.py --batch-size 64 --out-root $RUN_DIR --inputs $SENS